# Handling the imbalance on logistic regression

force the model to pay attention to classes 1 and 2, with 2 techniques, and measure what it changes. All resampling is applied to the **training set only**, after the split: the test set keeps its real, imbalanced distribution.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from src.config import SEED
from src.data import load_splits
from src.metrics import evaluate

sns.set_theme(style='whitegrid')

data, y_train, y_test = load_splits()

## The techniques

- **Class weights** (`class_weight='balanced'`): the data stays as is, but an error on a rare class costs more in the loss. Weight of class c = n / (5 × n_c).
- **Random undersampling**: randomly drop rows of the large classes until all classes have the size of the smallest one.
- **None**: the baseline of the previous notebook.

In [ ]:
n = len(y_train)
weights = n / (5 * y_train.value_counts().sort_index())
weights.round(3)

In [ ]:
X = data['no_salary']['X_train_s']
for name, sampler in {'undersampling': RandomUnderSampler(random_state=SEED)}.items():
    _, y_res = sampler.fit_resample(X, y_train)
    print(name, y_res.value_counts().sort_index().tolist())

## Training and evaluation

Four variants on both feature sets. The test set is untouched.

In [ ]:
def make(method):
    if method == 'class_weight':
        return None, LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
    sampler = {'none': None,
               'undersampling': RandomUnderSampler(random_state=SEED),
               }[method]
    return sampler, LogisticRegression(max_iter=1000, random_state=SEED)

methods = ['none', 'class_weight', 'undersampling']
rows, per_class = [], {}
for fs, d in data.items():
    for method in methods:
        sampler, model = make(method)
        X_tr, y_tr = d['X_train_s'], y_train
        if sampler is not None:
            X_tr, y_tr = sampler.fit_resample(X_tr, y_tr)
        pred = model.fit(X_tr, y_tr).predict(d['X_test_s'])
        summary, table = evaluate(y_test, pred)
        per_class[(fs, method)] = table
        rows.append({'features': fs, 'method': method, **summary,
                     'recall_1': table.loc[1, 'recall'], 'recall_2': table.loc[2, 'recall'],
                     'precision_1': table.loc[1, 'precision'], 'precision_2': table.loc[2, 'precision'],
                     'predicted_1': int((pred == 1).sum()), 'predicted_2': int((pred == 2).sum())})

results = pd.DataFrame(rows).set_index(['features', 'method'])
results.round(3)

## Recall per class

In [ ]:
recall = pd.DataFrame({k: v['recall'] for k, v in per_class.items()}).T
recall.columns = [f'class {c}' for c in recall.columns]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, fs in zip(axes, data):
    recall.loc[fs].plot.bar(ax=ax, rot=0)
    ax.set_title(fs)
    ax.set_xlabel('')
axes[0].set_ylabel('recall')
plt.tight_layout()
plt.show()

## Why macro F1 does not follow the recall of class 1

Predicting class 1 more often finds more of the 80 true rows (recall goes up), but most of the predictions are wrong (precision stays tiny), so the F1 of the class stays low.

In [ ]:
results[['recall_1', 'precision_1', 'predicted_1']].round(3).assign(true_class_1=int((y_test == 1).sum()))